# INTRODUCTION

The objective of this exercice is to **control the position of a 2D pendulum** using Jiminy simulator (https://github.com/duburcqa/jiminy).

The pole itself is massless, and all the weight is concentrated on the mass at the tip of the pendulum. The actual system is 3D, but its mechanical deisgn restricts its motion to plan. 

This pendulum has a single actuated joint on its root. It also features a bunch of sensors, i.e.:
* 1 encoder measuring the angle and velocity of the motor
* 1 effort sensor measuring the effort on the output of the motor (minus the friction in the transition due to a limitation of the simulator)
* 1 IMU sensor gathering 1 gyroscope and 1 accelerometer, measuring respectively the 3D angular velocity of the mass and the 3D acceleration of the mass minus gravity in local frame

A random desired mass angle is sampled at the beginning of each episode. The objective is to **reach this desired angle and maintain it without moving within a maximum time window of 3 seconds**. At the end of this time window, the norm of the linear velocity of the mass must not exceed 2e-3 m.s^-1, while the desired angle must be reached with a tolerance of 5e-3 rad.

It is proposed to solve this task by training a control policy capable using  Reinforcement Learning. One must consider that the agent is trained in simulation and later deployed on a real robotic arm. As a result, it is forbidden to pass anything else than a by-product of sensor measurements as observation after training but there is no such limitation during training.

# PROBLEM DEFINITION

Some files are going to be downloaded. They are required for running the simulation. They should be treated as implementation details. **Please refrain from opening them to re-engineer the behaviour of the system.**

In [ ]:
# Setup the environment for google colab (make sure to select "T4 GPU" runtime type beforehand).
# Note that it is necessary to restart the session after executing this cell for the first time.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    !pip install ipympl jiminy_py[meshcat] gym_jiminy[all] stable_baselines3 > /dev/null 2>&1
    !wget https://raw.githubusercontent.com/duburcqa/jiminy/demo/simple_pendulum/robot.urdf -O robot.urdf > /dev/null 2>&1
    !wget https://raw.githubusercontent.com/duburcqa/jiminy/demo/simple_pendulum/robot_hardware.toml -O robot_hardware.toml > /dev/null 2>&1
    !wget https://raw.githubusercontent.com/duburcqa/jiminy/demo/simple_pendulum/robot_options.toml -O robot_options.toml > /dev/null 2>&1
    !wget https://raw.githubusercontent.com/duburcqa/jiminy/demo/simple_pendulum/part_4.toml -O part_4.toml > /dev/null 2>&1
except ImportError:
    pass

In [ ]:
# Enable matplotlib interactive notebook integration
%matplotlib widget

In [ ]:
import typing as tp

# Generic import that will be useful throughout this notebook
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# The core modules of jiminy and its RL extension gym_jiminy
# Official doc: https://duburcqa.github.io/jiminy/api/jiminy_py/index.html
from jiminy_py.simulator import Simulator
from gym_jiminy.common.bases import EngineObsType
from gym_jiminy.common.envs import BaseJiminyEnv
from gym_jiminy.common.utils import sample

# Import low-level Rigid-Body Dynamics library
import pinocchio as pin

# Define the learning environment
class PendulumEnv(BaseJiminyEnv):
    def __init__(self, step_dt: float, render_mode: tp.Optional[str] = None):
        # Instantiate the simulator
        simulator = Simulator.build(
            urdf_path="robot.urdf",
            has_freeflyer=False,
            viewer_kwargs=dict(
                camera_pose=([-5.0, 0.0, 0.0], [np.pi/2, 0.0, -np.pi/2])
            ),
        )
        robot = simulator.robot

        # --------------------------------------------------------------
        # TODO: Uncomment this line if you are ready for extra challenge
        # simulator.import_options("part_4.toml")
        # --------------------------------------------------------------

        # Allocate memory for the desired mass angle
        self.angle_desired = np.array(0.0)

        # Define the current mass angle and velocity
        self._frame_idx = env.robot.pinocchio_model.getFrameId('PendulumMass')
        self.velocity = np.array(0.0)
        self.angle = np.array(0.0)

        # Initialize base environment
        super().__init__(
            simulator=simulator,
            step_dt=step_dt,
            simulation_duration_max=3.0,
            render_mode=render_mode
        )

        # Add the desired mass angle to the observation space
        self.observation['states']['task'] = self.angle_desired

    def _setup(self) -> None:
        # Call base implementation
        super()._setup()

        # Randomly sample a new desired mass angle
        self.angle_desired[()] = sample(scale=np.pi, rg=self.np_random)

    def _initialize_observation_space(self) -> None:
        # Call base implementation
        super()._initialize_observation_space()

        # Add the desired mass angle to the original observation space
        self.observation_space['states']['task'] = gym.spaces.Box(
            low=np.array(-np.pi), high=np.array(np.pi), dtype=np.float64
        )

    def refresh_observation(self, measurement: EngineObsType) -> None:
        super().refresh_observation(measurement)

    def _refresh_buffers(self) -> None:
        v_spatial = pin.getFrameVelocity(
            self.robot.pinocchio_model, self.robot.pinocchio_data, frame_idx
        )
        self.velocity[()] = np.linalg.norm(v_spatial.linear)

        mass_position = self.robot.pinocchio_data.oMf[frame_idx].translation
        self.angle[()] = np.arctan2(-mass_position[1], mass_position[2])

    def _sample_state(self):
        q_init_th = sample(
            low=self.robot.pinocchio_model_th.lowerPositionLimit,
            high=self.robot.pinocchio_model_th.upperPositionLimit,
            rg=self.np_random,
        )
        q_init_th = pin.normalize(self.robot.pinocchio_model_th, q_init_th)
        q_init = self.robot.get_extended_position_from_theoretical(q_init_th)

        v_init_th = 2 * sample(
            shape=(self.robot.pinocchio_model_th.nv,),
            rg=self.np_random,
        )
        v_init = self.robot.get_extended_velocity_from_theoretical(v_init_th)
        return q_init, v_init

    def has_terminated(self, info):
        # --------------------------------------------------------------
        # TODO: Implement your own termination condition if necessary
        terminated, truncated = False, False
        # --------------------------------------------------------------

        return terminated, truncated

    def compute_reward(self, terminated, info):
        # --------------------------------------------------------------
        # TODO: Implement your own reward if necessary
        reward = 0.0
        # --------------------------------------------------------------

        return reward

Add some pre- and post-processing layers to the base environment.

The following pipeline is proposed:
* Extract only sensors measurements and task (i.e. desired angle of the mass) from the original observation space as it contains additional information that cannot be observed on a real robot (see next cell).
* Flatten the nexted multi-dimensional observation space as a 1D vector
* Normalize the action space

This pipeline can be modified freely but it is unlikely to be necessary.

In [ ]:
from jiminy_py.viewer import Viewer

from gym_jiminy.common.wrappers import (
    FilterObservation, NormalizeAction, FlattenObservation
)

# The environment timestep.
# This parameter can be modified freely but it is unlikely to be necessary.
STEP_DT = 0.02

env_creator = lambda *args, **kwargs : (
    NormalizeAction(
        FlattenObservation(
                FilterObservation(
                    PendulumEnv(step_dt=STEP_DT, render_mode="rgb_array"),
                    nested_filter_keys=(
                        "measurements",
                        ("states", "task"),
                    )
                )
            )
        )
    )

# EVALUATION FOR RANDOM POLICY

In [ ]:
# Instantiate the environment
# Note that 'eval' mode must be enabled for replay and plot to work properly.
Viewer.close()
env = env_creator()
env.eval()

In [ ]:
# Print the original observation space (before pipeline processing)
env.unwrapped.observation_space

In [ ]:
# Print the original sensor measurements
env.robot.sensor_measurements

In [ ]:
# Sample one episode using a random policy until termination (or truncation due to time limit)
observation, info = env.reset()
terminated, truncated = False, False
while not (terminated or truncated):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
env.stop()

In [ ]:
# Replay the episode
env.replay()

In [ ]:
# Plot all simulation data
env.plot()

# TRAINING

It is proposed to train a basic MLP policy using Stable Baselines3 library for simplicity. Any other library can be used without restriction.

Default parameters are provided as a starting point. They are meaningful initial guess but it may be necessary to modify them.

In [ ]:
from torch import nn
from stable_baselines3.ppo import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import (
    EvalCallback, StopTrainingOnRewardThreshold as StopOnReward)

# Agent algorithm config
config = {}
config['n_steps'] = 4000
config['batch_size'] = 250
config['learning_rate'] = 5.0e-4
config['n_epochs'] = 20
config['gamma'] = 0.98
config['gae_lambda'] = 0.94
config['target_kl'] = 0.1
config['ent_coef'] = 0.01
config['vf_coef'] = 0.04
config['clip_range'] = 0.3
config['clip_range_vf'] = None
config['max_grad_norm'] = 1.0
config['seed'] = 0

# Policy model config
config['policy_kwargs'] = {
    'net_arch': dict(pi=[64, 64], vf=[64, 64]),
    'activation_fn': nn.Tanh,
    'ortho_init': True,
    'log_std_init': 1.0,
    'optimizer_kwargs': {
        'weight_decay': 1e-4,
        'betas': (0.9, 0.999),
        'eps': 1e-6,
    }
}

# Create a multiprocess environment
train_env = make_vec_env(
    env_creator, n_envs=4, vec_env_cls=SubprocVecEnv, seed=0)
test_env = make_vec_env(
    env_creator, n_envs=1, vec_env_cls=DummyVecEnv, seed=0)

# Create the learning agent according to the chosen algorithm
train_agent = PPO('MlpPolicy', train_env, **config, device='cpu', verbose=True)

# Create callback to stop learning early if reward threshold is exceeded
callback_reward = StopOnReward(reward_threshold=600)
eval_callback = EvalCallback(
    test_env, callback_on_new_best=callback_reward,
    eval_freq=10000 // train_agent.n_envs, n_eval_episodes=10,
    verbose=True)

# Run the learning process
train_agent.learn(total_timesteps=400000, callback=eval_callback)

# EVALUATION FOR TRAINED POLICY

In [ ]:
import time
from jiminy_py.viewer import sleep

# Make sure that the task can be achieved successfully several times in a row
for seed in range(10):
    # Sample a new episode.
    # Note that the episode will last 3s no later what. The condition to check
    # if the task was successfully achieved will be checked after this duration
    # no matter if the agent was faster than this.
    observation, info = env.reset(seed=seed)
    frame_idx = env.robot.pinocchio_model.getFrameId('PendulumMass')
    while env.stepper_state.t < 3.0:
        action, _ = train_agent.predict(observation, deterministic=True)
        observation, _, _, _, _ = env.step(action)
    env.stop()

    # Check that the mass reached the desired angle and is not moving anymore
    assert np.allclose(env.unwrapped.velocity, 0.0, atol=1e-3), (
        "Pendulum has not reached equilibrium"
    )
    angle = env.unwrapped.angle
    angle_desired = env.unwrapped.angle_desired
    error_angle = min(
        np.abs(angle - angle_desired),
        2.0 * np.pi - np.abs(angle - angle_desired),
    )
    assert error_angle < 1e-3, (
        f"Error is too large: measured={angle:.3f}, target={angle_desired:.3f}"
    )

In [ ]:
# Replay the episode
env.replay()

In [ ]:
# Plot all simulation data
env.plot()